In [6]:
"""
Toric code on an Lx x Ly square lattice (periodic boundary conditions),
qubits on edges, with transverse fields in BOTH x and z directions:

H = -Je * sum_s A_s  - Jm * sum_p B_p  - hx * sum_e X_e  - hz * sum_e Z_e
A_s = product_{edges touching vertex s} X_e
B_p = product_{edges around plaquette p} Z_e

WARNING: This builds the FULL 2^N x 2^N matrix (sparse), so it only works
for small N = 2*Lx*Ly (e.g. N <= ~16–20 depending on RAM).
"""

from dataclasses import dataclass
import numpy as np
import scipy.sparse as sp


# -------------------------
# Pauli matrices (sparse)
# -------------------------
I2 = sp.identity(2, format="csr", dtype=np.complex128)
X2 = sp.csr_matrix(np.array([[0, 1], [1, 0]], dtype=np.complex128))
Y2 = sp.csr_matrix(np.array([[0, -1j], [1j, 0]], dtype=np.complex128))
Z2 = sp.csr_matrix(np.array([[1, 0], [0, -1]], dtype=np.complex128))

PAULI = {"I": I2, "X": X2, "Y": Y2, "Z": Z2}


def kron_all(ops):
    """Kronecker product of a list of 2x2 sparse ops -> (2^N)x(2^N) sparse."""
    out = ops[0]
    for op in ops[1:]:
        out = sp.kron(out, op, format="csr")
    return out


def pauli_string(N, where_to_pauli):
    """
    Build a Pauli string on N qubits as sparse matrix.

    where_to_pauli: dict {q: 'X'/'Y'/'Z'} specifying which qubits are non-identity.
    """
    ops = [I2] * N
    for q, p in where_to_pauli.items():
        ops[q] = PAULI[p]
    return kron_all(ops)


# -------------------------
# Lattice + indexing
# -------------------------
@dataclass(frozen=True)
class ToricLattice:
    Lx: int
    Ly: int

    @property
    def Nq(self) -> int:
        # qubits on edges: horizontal + vertical = 2*Lx*Ly
        return 2 * self.Lx * self.Ly

    def _wrap(self, x, y):
        return (x % self.Lx, y % self.Ly)

    def q_h(self, x, y) -> int:
        """
        Qubit index for HORIZONTAL edge from (x,y) -> (x+1,y).
        """
        x, y = self._wrap(x, y)
        return x + self.Lx * y  # 0 .. Lx*Ly-1

    def q_v(self, x, y) -> int:
        """
        Qubit index for VERTICAL edge from (x,y) -> (x,y+1).
        """
        x, y = self._wrap(x, y)
        return self.Lx * self.Ly + (x + self.Lx * y)  # Lx*Ly .. 2*Lx*Ly-1

    def star_edges(self, x, y):
        """
        The 4 edges touching vertex (x,y):
          - horizontal leaving (x,y): h(x,y)
          - horizontal entering from left: h(x-1,y)
          - vertical leaving (x,y): v(x,y)
          - vertical entering from below: v(x,y-1)
        """
        return [
            self.q_h(x, y),
            self.q_h(x - 1, y),
            self.q_v(x, y),
            self.q_v(x, y - 1),
        ]

    def plaquette_edges(self, x, y):
        """
        The 4 edges around plaquette with lower-left corner at (x,y):
          bottom: h(x,y)
          right:  v(x+1,y)
          top:    h(x,y+1)
          left:   v(x,y)
        Orientation doesn't matter for Z-products.
        """
        return [
            self.q_h(x, y),
            self.q_v(x + 1, y),
            self.q_h(x, y + 1),
            self.q_v(x, y),
        ]


# -------------------------
# Hamiltonian builder
# -------------------------
def toric_code_hamiltonian(Lx, Ly, Je=1.0, Jm=1.0, hx=0.0, hz=0.0, dtype=np.complex128):
    lat = ToricLattice(Lx, Ly)
    N = lat.Nq
    dim = 2 ** N
    H = sp.csr_matrix((dim, dim), dtype=dtype)

    # -Je * sum_s A_s  (A_s is product of X on the 4 incident edges)
    for x in range(Lx):
        for y in range(Ly):
            edges = lat.star_edges(x, y)
            H += (-Je) * pauli_string(N, {q: "X" for q in edges})

    # -Jm * sum_p B_p  (B_p is product of Z on the 4 boundary edges)
    for x in range(Lx):
        for y in range(Ly):
            edges = lat.plaquette_edges(x, y)
            H += (-Jm) * pauli_string(N, {q: "Z" for q in edges})

    # -hx * sum_e X_e  -hz * sum_e Z_e
    if hx != 0.0:
        for q in range(N):
            H += (-hx) * pauli_string(N, {q: "X"})
    if hz != 0.0:
        for q in range(N):
            H += (-hz) * pauli_string(N, {q: "Z"})

    return H, lat


import numpy as np

def group_degeneracies(evals, tol=1e-10):
    """
    Group eigenvalues into degenerate multiplets.
    Returns a list of dicts: [{"E":..., "mult":..., "indices":(i0,i1)}, ...]
    where indices refer to sorted eigenvalues slice [i0:i1].
    """
    evals = np.asarray(evals, dtype=float)
    order = np.argsort(evals)
    e = evals[order]

    groups = []
    i = 0
    n = len(e)
    while i < n:
        j = i + 1
        # grow cluster while energies are within tol of the cluster start
        while j < n and abs(e[j] - e[i]) <= tol:
            j += 1
        groups.append({
            "E": float(np.mean(e[i:j])),
            "mult": int(j - i),
            "indices": (int(i), int(j)),
            "members": e[i:j],   # actual values in the cluster
        })
        i = j
    # sort groups by energy (already) and keep order mapping if needed
    return groups

def degeneracy_report(evals, tol=1e-10, top_k=15):
    groups = group_degeneracies(evals, tol=tol)
    max_mult = max(g["mult"] for g in groups)
    num_degenerate_levels = sum(1 for g in groups if g["mult"] > 1)

    print(f"Total states: {len(evals)}")
    print(f"Unique energy levels (within tol={tol:g}): {len(groups)}")
    print(f"Levels with multiplicity > 1: {num_degenerate_levels}")
    print(f"Max multiplicity: {max_mult}")

    # show the most degenerate levels
    groups_sorted = sorted(groups, key=lambda g: (-g["mult"], g["E"]))
    print("\nMost degenerate energy levels:")
    for g in groups_sorted[:top_k]:
        E = g["E"]
        m = g["mult"]
        print(f"  E ≈ {E:+.12f}   mult = {m}")

    return groups

def full_spectrum_dense(H):
    """
    Full diagonalization (dense). Works only for small Hilbert space.
    H can be sparse; it will be densified.
    """
    Hd = H.toarray() if hasattr(H, "toarray") else np.asarray(H)
    # For Hermitian matrices:
    evals = np.linalg.eigvalsh(Hd)
    return evals

# Example usage with your toric_code_hamiltonian builder:
if __name__ == "__main__":
    # assumes you already have toric_code_hamiltonian(Lx, Ly, ...)
    Lx, Ly = 2, 2  # N=8 -> dim=256 (easy)
    H, lat = toric_code_hamiltonian(Lx, Ly, Je=1.0, Jm=1.0, hx=0.0, hz=0.0)

    evals = full_spectrum_dense(H)
    groups = degeneracy_report(evals, tol=1e-10, top_k=20)

    # Now turn on a perturbation and see degeneracies lift
    H2, _ = toric_code_hamiltonian(Lx, Ly, Je=1.0, Jm=1.0, hx=0.02, hz=0.07)
    evals2 = full_spectrum_dense(H2)
    print("\nWith hx,hz turned on:")
    groups2 = degeneracy_report(evals2, tol=1e-12, top_k=20)


Total states: 256
Unique energy levels (within tol=1e-10): 5
Levels with multiplicity > 1: 5
Max multiplicity: 152

Most degenerate energy levels:
  E ≈ -0.000000000000   mult = 152
  E ≈ -4.000000000000   mult = 48
  E ≈ +4.000000000000   mult = 48
  E ≈ -8.000000000000   mult = 4
  E ≈ +8.000000000000   mult = 4

With hx,hz turned on:
Total states: 256
Unique energy levels (within tol=1e-12): 137
Levels with multiplicity > 1: 90
Max multiplicity: 11

Most degenerate energy levels:
  E ≈ -0.000000000000   mult = 11
  E ≈ -0.145602197786   mult = 9
  E ≈ +0.145602197786   mult = 9
  E ≈ -4.009800928426   mult = 3
  E ≈ -4.000800980698   mult = 3
  E ≈ -0.221674728526   mult = 3
  E ≈ -0.111231334991   mult = 3
  E ≈ +0.114408243426   mult = 3
  E ≈ +0.229099729215   mult = 3
  E ≈ -4.280789139359   mult = 2
  E ≈ -4.089907687549   mult = 2
  E ≈ -4.014776301625   mult = 2
  E ≈ -4.014705913610   mult = 2
  E ≈ -4.009902465430   mult = 2
  E ≈ -4.009800927476   mult = 2
  E ≈ -4.0051219